# Notebook 05 — Robustness Checks

Validates the DriftFire regime>=3 strategy against cost sensitivity, SPY benchmark, monthly return patterns, and a statistical significance test.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf
import warnings, time
warnings.filterwarnings('ignore')

from backtest import run_backtest, calc_metrics
from report_utils import (set_plot_style, plot_drawdown, plot_trade_distribution,
                          plot_monthly_heatmap, plot_cost_sensitivity)

DATA_DIR = '../data/processed/'
FIG_DIR  = '../reports/figures/'

INITIAL_CAPITAL = 100_000
MAX_POSITIONS   = 5
POSITION_SIZE   = 0.20
MAX_HOLD_DAYS   = 20
TRAIN_END  = '2020-12-31'
VAL_END    = '2021-12-31'
TEST_START = '2022-01-01'

_nb_start = time.time()

df     = pd.read_parquet(DATA_DIR + 'features.parquet')
trades = pd.read_parquet(DATA_DIR + 'backtest_trades.parquet')
equity = pd.read_parquet(DATA_DIR + 'backtest_equity.parquet')

df['date'] = pd.to_datetime(df['date'])
trades['entry_date'] = pd.to_datetime(trades['entry_date'])
trades['exit_date']  = pd.to_datetime(trades['exit_date'])
equity['date']       = pd.to_datetime(equity['date'])

# Add period to trades
def get_period(d):
    d = pd.Timestamp(d)
    if d <= pd.Timestamp(TRAIN_END): return 'train'
    elif d <= pd.Timestamp(VAL_END): return 'val'
    return 'test'

if 'period' not in trades.columns:
    trades['period'] = trades['entry_date'].apply(get_period)

print(f'Features: {len(df):,} rows')
print(f'Trades (regime>=3): {len(trades)}')
print(f'Equity rows: {len(equity)}')


/Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Features: 823,610 rows
Trades (regime>=3): 412
Equity rows: 1760


## 5.1 — Transaction Cost Sensitivity

In [2]:
print('Running cost sensitivity (regime>=3)...')
cost_results = []
for c in [5, 10, 15, 20, 30, 50]:
    t_c, e_c = run_backtest(df, signal_col='entry_regime3', cost_bps=c)
    m = calc_metrics(t_c, e_c, label='full', signal=f'{c}bps')
    cost_results.append({'cost_bps': c, 'sharpe': m['sharpe'], 'cagr_%': m['cagr_%'], 'max_dd_%': m['max_dd_%']})

cost_df = pd.DataFrame(cost_results)
print(cost_df.to_string(index=False))

fig = plot_cost_sensitivity(
    cost_df, baseline_bps=10,
    title='Sharpe vs Transaction Cost — regime>=3',
    save_path=FIG_DIR + 'cost_sensitivity.png'
)
plt.show()
print('Saved: cost_sensitivity.png')


Running cost sensitivity (regime>=3)...


 cost_bps  sharpe  cagr_%  max_dd_%
        5    0.22    -0.1     -30.0
       10    0.22    -0.2     -30.1
       15    0.22    -0.2     -30.1
       20    0.22    -0.2     -30.2
       30    0.22    -0.3     -30.3
       50    0.21    -0.4     -30.5
Saved: cost_sensitivity.png


## 5.2 — SPY Benchmark Comparison

In [3]:
print('Downloading SPY for benchmark comparison...')
spy_raw = yf.download('SPY', start='2018-01-01', end='2025-01-01', progress=False)
spy_raw.index = pd.to_datetime(spy_raw.index)
if isinstance(spy_raw.columns, pd.MultiIndex):
    spy_close = spy_raw[('Close', 'SPY')]
else:
    spy_close = spy_raw['Close']
spy_close = spy_close.squeeze()

def spy_metrics(start, end, label):
    sub = spy_close[(spy_close.index >= start) & (spy_close.index <= end)]
    if len(sub) < 2: return {}
    dr = sub.pct_change().dropna()
    n_years = (sub.index[-1] - sub.index[0]).days / 365.25
    cagr   = (sub.iloc[-1] / sub.iloc[0]) ** (1 / n_years) - 1
    sharpe = dr.mean() / dr.std() * np.sqrt(252)
    maxdd  = ((sub - sub.cummax()) / sub.cummax()).min()
    return {'period': label, 'sharpe': round(sharpe, 2), 'cagr_%': round(cagr*100,1), 'max_dd_%': round(maxdd*100,1)}

spy_bench = pd.DataFrame([
    spy_metrics('2018-01-01', TRAIN_END, 'train'),
    spy_metrics('2021-01-01', VAL_END,   'val'),
    spy_metrics(TEST_START,   '2024-12-31', 'test'),
    spy_metrics('2018-01-01', '2024-12-31', 'full'),
])

# Strategy metrics for regime>=3
strat_bench = pd.read_csv(DATA_DIR + 'backtest_comparison.csv')
strat_bench = strat_bench[strat_bench['signal'] == 'regime>=3'][['period', 'sharpe', 'cagr_%', 'max_dd_%']].dropna()

print('='*60)
print('SPY Buy-and-Hold:')
print(spy_bench.to_string(index=False))
print('\nDriftFire regime>=3:')
print(strat_bench.to_string(index=False))

SPY Buy-and-Hold:
period  sharpe  cagr_%  max_dd_%
 train    0.68    13.8     -33.7
   val    2.13    30.9      -5.1
  test    0.56     8.7     -24.5
  full    0.75    13.6     -33.7

DriftFire regime>=3:
Empty DataFrame
Columns: [period, sharpe, cagr_%, max_dd_%]
Index: []


## 5.3 — Monthly Returns Heatmap

In [4]:
fig = plot_monthly_heatmap(
    equity,
    title='Monthly Returns Heatmap — regime>=3 Strategy',
    save_path=FIG_DIR + 'monthly_returns.png'
)
plt.show()
print('Saved: monthly_returns.png')


Saved: monthly_returns.png


## 5.4 — t-test: Signal Days vs Random Non-Signal Days

In [5]:
# Compute 5-day forward return for every stock-day
df_stk = df[df['ticker'].notna()].copy()
df_stk = df_stk.sort_values(['ticker', 'date'])
df_stk['fwd_5d_ret'] = df_stk.groupby('ticker')['close'].pct_change(-5)  # negative shift = forward

signal_mask = df_stk['entry_regime3'] == True
non_sig_mask = (df_stk['entry_signal'] == False) & df_stk['fwd_5d_ret'].notna()

signal_rets  = df_stk[signal_mask & df_stk['fwd_5d_ret'].notna()]['fwd_5d_ret'] * 100
non_sig_pool = df_stk[non_sig_mask]['fwd_5d_ret'] * 100

# Sample same size as signal group from non-signal days
np.random.seed(42)
n_sample = len(signal_rets)
non_sig_sample = non_sig_pool.sample(n=min(n_sample * 10, len(non_sig_pool)), random_state=42)

t_stat, p_val = stats.ttest_ind(signal_rets, non_sig_sample, equal_var=False)

print(f'Signal group (entry_regime3):  n={len(signal_rets)}, mean={signal_rets.mean():.2f}%, std={signal_rets.std():.2f}%')
print(f'Non-signal group (random):     n={len(non_sig_sample)}, mean={non_sig_sample.mean():.2f}%, std={non_sig_sample.std():.2f}%')
print(f'\nt-statistic: {t_stat:.3f}')
print(f'p-value:     {p_val:.4f}')
if p_val < 0.05:
    print('Result: SIGNIFICANT at 5% level — signal days have different forward returns')
elif p_val < 0.10:
    print('Result: Marginally significant at 10% level')
else:
    print('Result: NOT significant — forward returns not statistically different')
    print('(Small sample size may limit power of the test)')

# Bar chart: mean forward returns with error bars
means = [signal_rets.mean(), non_sig_sample.mean()]
sems  = [signal_rets.sem(), non_sig_sample.sem()]
labels = ['Signal Days\n(entry_regime3)', 'Non-Signal Days\n(random sample)']
colors_bar = ['steelblue', 'lightgray']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=sems, capsize=6, color=colors_bar, edgecolor='black', width=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('5-Day Forward Return (%)')
ax.set_title(f'Event Study: Signal vs Non-Signal Forward Returns\np={p_val:.3f}, t={t_stat:.2f}', fontsize=11)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + (0.05 if mean >= 0 else -0.15),
            f'{mean:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR + 'event_study.png', dpi=150)
plt.show()
print('Saved: event_study.png')

Signal group (entry_regime3):  n=19, mean=0.88%, std=4.23%
Non-signal group (random):     n=190, mean=0.44%, std=4.28%

t-statistic: 0.435
p-value:     0.6679
Result: NOT significant — forward returns not statistically different
(Small sample size may limit power of the test)
Saved: event_study.png


## 5.5 — Regime Score Breakdown Table

In [6]:
# Show what regime_score level was present at each entry_regime3 signal
regime3_signals = df[df['entry_regime3'] == True][['date', 'ticker', 'regime_score', 'vol_ratio', 'sector_etf']].copy()

print('Regime score breakdown at entry_regime3 signal days:')
print(regime3_signals['regime_score'].value_counts().sort_index())

# Hit rate by regime_score (merge with trades)
trades['regime_score'] = trades['entry_date'].map(
    df[df['entry_regime3']].drop_duplicates('date').set_index('date')['regime_score']
)

print('\nTrade outcomes by regime_score:')
breakdown = trades.groupby('regime_score').agg(
    n_trades=('return_pct', 'count'),
    hit_rate=('return_pct', lambda x: (x > 0).mean() * 100),
    avg_return=('return_pct', lambda x: x.mean() * 100),
    avg_days=('days_held', 'mean')
).round(2)
print(breakdown.to_string())

# Distribution of signals by sector
print('\nSignals by sector ETF:')
print(regime3_signals['sector_etf'].value_counts())

Regime score breakdown at entry_regime3 signal days:
regime_score
3     2
4    17
Name: count, dtype: int64

Trade outcomes by regime_score:
              n_trades  hit_rate  avg_return  avg_days
regime_score                                          
3.0                  2       0.0       -2.73       4.0
4.0                  5      20.0       -1.11       5.4

Signals by sector ETF:
sector_etf
XLK     6
XLV     2
XLI     2
XLB     2
XLU     2
XLY     2
XLC     1
XLF     1
XLRE    1
Name: count, dtype: int64


## 5.6 — Confirm All Figures

In [7]:
import os
expected_figs = [
    'signal_frequency.png', 'equity_curve.png', 'drawdown.png',
    'trade_returns_dist.png', 'cost_sensitivity.png', 'monthly_returns.png', 'event_study.png'
]
print('Figure files in reports/figures/:')
for f in expected_figs:
    path = FIG_DIR + f
    exists = os.path.exists(path)
    size = os.path.getsize(path) // 1024 if exists else 0
    print(f'  {"OK" if exists else "MISSING":8s}  {f}  ({size} KB)')

print('\nBacktest output files:')
for fname in ['backtest_trades.parquet', 'backtest_equity.parquet', 'backtest_comparison.csv', 'features.parquet']:
    path = DATA_DIR + fname
    exists = os.path.exists(path)
    size = os.path.getsize(path) // 1024 if exists else 0
    print(f'  {"OK" if exists else "MISSING":8s}  {fname}  ({size} KB)')

print('\nAll robustness checks complete. Proceed to notebook 06 for the report.')

Figure files in reports/figures/:
  OK        signal_frequency.png  (43 KB)
  OK        equity_curve.png  (223 KB)
  OK        drawdown.png  (150 KB)
  OK        trade_returns_dist.png  (65 KB)
  OK        cost_sensitivity.png  (42 KB)
  OK        monthly_returns.png  (124 KB)
  OK        event_study.png  (46 KB)

Backtest output files:
  OK        backtest_trades.parquet  (37 KB)
  OK        backtest_equity.parquet  (24 KB)
  OK        backtest_comparison.csv  (1 KB)
  OK        features.parquet  (139010 KB)

All robustness checks complete. Proceed to notebook 06 for the report.
